In [1]:
# misc_df=pd.read_excel(r'training_data_oversampled_updated.xlsx')
# misc_df.head()

In [2]:
# misc_df.drop_duplicates(inplace=True)

In [3]:
# misc_df['DOCUMENT_TYPE'].value_counts()

In [4]:
# misc_df['DOCUMENT_TYPE']=misc_df['DOCUMENT_TYPE'].apply(int)

In [5]:
# len(misc_df)

In [6]:
# misc_df.loc[misc_df['DOCUMENT_TYPE']==0]

In [7]:
# misc_df=misc_df.loc[misc_df['DOCUMENT_TYPE']==0]

In [8]:
# misc_df

In [9]:
# misc_df.to_excel(r'misc_df.xlsx',index=False)

In [10]:
!nvidia-smi

Wed Feb 15 18:59:52 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 440.33.01    Driver Version: 440.33.01    CUDA Version: 10.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|===============================+======================+======================|
|   0  Tesla V100-SXM2...  On   | 00000000:00:1B.0 Off |                    0 |
| N/A   46C    P0    55W / 300W |   1372MiB / 16160MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
|   1  Tesla V100-SXM2...  On   | 00000000:00:1C.0 Off |                    0 |
| N/A   43C    P0    38W / 300W |     11MiB / 16160MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
|   2  T

In [11]:
import json
import torch
import torch.nn as nn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re
import spacy
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import string
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import mean_squared_error, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.corpus import stopwords
import random
import itertools
import os
from modelsAndDatasets import LSTMNetLatest, LSTMNet, DocumentDataset, LSTM_fixed_len, LstmW2VNet

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [12]:
# !nvidia-smi

In [13]:
def validation_metrics(model, valid_dl):
    model.eval()
    correct = 0
    total = 0
    sum_loss = 0.0
    sum_rmse = 0.0
    for x, y, l in valid_dl:
        x = x.long()
        y = y.long()
        y_hat = model(x, l)
        loss = F.cross_entropy(y_hat, y)
        pred = torch.max(y_hat, 1)[1]
        correct += (pred == y).float().sum()
        total += y.shape[0]
        sum_loss += loss.item() * y.shape[0]
        sum_rmse += np.sqrt(mean_squared_error(pred, y.unsqueeze(-1))) * y.shape[0]
    return sum_loss / total, correct / total, sum_rmse / total


def get_confusion_matrix(model, valid_dl, nb_classes=2):
    matrix = torch.zeros(nb_classes, nb_classes)
    with torch.no_grad():
        for inputs, classes, i in valid_dl:
            inputs = inputs.to(device)
            classes = classes.to(device)
            outputs = model(inputs, 1)
            _, preds = torch.max(outputs, 1)

            for t, p in zip(classes.view(-1), preds.view(-1)):
                matrix[t.long(), p.long()] += 1

    print(matrix)
    plt.figure(figsize=(15, 10))
    class_names = [1, 2, 3, 5, 6, 12]
    df_cm = pd.DataFrame(matrix, index=class_names, columns=class_names).astype(int)
    heatmap = sns.heatmap(df_cm, annot=True, fmt="d")

    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=0, ha='right', fontsize=15)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=45, ha='right', fontsize=15)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    return matrix



In [14]:
def getModel(classifier_path):
    with open('vector_v1.txt') as f:
        data = f.read()
    vector_config = json.loads(data)
    print('vector_config loaded.')

    vocab2index = vector_config['vocab2index']
    mean_words = vector_config['mean_words']
    vocab_size = vector_config['vocab_size']
    embedding_dim = vector_config['embedding_dim']
    hidden_dim = vector_config['hidden_dim']
    n_classes = vector_config['n_classes']

    model = LSTM_fixed_len(vocab_size, embedding_dim, hidden_dim, n_classes)
    # classifier_path = './model_correct_labels_embedding_100.pth'
    checkpoint = torch.load(classifier_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print('model loaded')
    return model


# Basic Pre-processing of text data
def nlp_pre_process_stem(df, targetColumnName, outputColumnName='text', to_csv=False, safety_rows=50000):
    tokenized_sentences = []
    stemmer = PorterStemmer()
    k=0
    for i in df.index:
        if type(df.iloc[i].text) != str:
            print(i)
            review = re.sub('[^a-zA-Z]', ' ', df.loc[i, targetColumnName])
            review = review.lower()
            review = review.split()
            review = [stemmer.stem(word) for word in review if
                      (word not in set(stopwords.words('english'))) and (len(word) > 2)]
            tokenized_sentences.append(' '.join(review))
            df.loc[i, outputColumnName] = tokenized_sentences[k]
            k = k + 1

            if to_csv and i % safety_rows == 0:
                df.to_csv('result_' + str(i) + '.csv', index=False)
            print(i)
        else:
            continue
    if to_csv:
        df.to_csv('final_clean_date.csv', index=False)
    return df, tokenized_sentences


def pre_process_stem_sentence_array(sentences):
    stemmer = PorterStemmer()
    tokenized_sentences = []
    for sentence in sentences:
        sentence = sentence.lower()
        sentence = sentence.split()
        sentence = [stemmer.stem(word) for word in sentence if
                    (word not in set(stopwords.words('english'))) and (len(word) > 2)]
        tokenized_sentences.append(' '.join(sentence))
    return tokenized_sentences


def pre_process_stem_sentence(sentence):
    stemmer = PorterStemmer()
    sentence = re.sub('[^a-zA-Z]', ' ', str(sentence))
    sentence = sentence.lower()
    sentence = sentence.split()
    sentence = [stemmer.stem(word) for word in sentence if
                (word not in set(stopwords.words('english'))) and (len(word) > 2)]
    sentence = ' '.join(sentence)
    return sentence


def classToLabels(y):
    zero_numbering = {0: 0, 27: 0, 1: 1, 2: 2, 3: 3, 5: 4, 6: 5, 12: 6, 25: 7, 51: 7, 30: 8}
    return y.apply(lambda x: zero_numbering[x])


# changing 0-numbering to original classes
def labelToClass(y):
    zero_numbering = {0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 6, 6: 12, 7: 25, 8: 30}
    return zero_numbering[y]


def encode_sentence(text, vocab2index, n_words=250):
    tokenized = text.split()
    encoded = np.zeros(n_words, dtype=int)
    enc1 = np.array([vocab2index.get(word, vocab2index["UNK"]) for word in tokenized])
    length = min(n_words, len(enc1))
    encoded[:length] = enc1[:length]
    return encoded, length


def classify_local(model, vocab2index, mean_words, text):
    # logger.info('Inside classify method')
    # logger.info(s3_url)
    if text is not None:
#         text = pre_process_stem_sentence(text)
        encoded_sentence = np.array(encode_sentence(text, vocab2index, mean_words))
        x = torch.from_numpy(encoded_sentence[0].astype(np.int32)).to(device)
        x = x.long()
        x = x.reshape(1, x.shape[0])
        y_hat = model(x, 1)
        pred = torch.max(y_hat, 1)[1]
        pred = pred.cpu().tolist()[0]
        pred = labelToClass(pred)
        return pred
    else:
        print('Empty text received for classification. Fatal Error. Please check OCR data of document - ')
        return 0
    # # test_transformer = transformer.fit_transform(counts_test)
    # y = model.predict(test_transformer)
    # return y.
    
def load_vector(vector_config_path):
    with open(vector_config_path) as f:
        temp = f.read()
    vector_config = json.loads(temp)
    print('vector_config loaded.')
    return vector_config['vocab2index'],vector_config['mean_words'], vector_config['vocab_size'], vector_config['embedding_dim'],vector_config['hidden_dim'],vector_config['n_classes']


def load_model(model_path, vector_config_path='vector_v1.txt', latest=True):
    
    vocab2index,mean_words,vocab_size,embedding_dim,hidden_dim,n_classes = load_vector(vector_config_path)
    
    if latest:
        model = LSTMNetLatest(vocab_size, embedding_dim, hidden_dim, n_classes)
    else:
        model = LSTMNet(vocab_size, embedding_dim, hidden_dim, n_classes)

    classifier_path = model_path
    checkpoint = torch.load(classifier_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print('model loaded')
    return model, vocab2index, mean_words

In [15]:
from tqdm import tqdm

In [16]:
# data = pd.read_excel('relevant_after_jan21_noURL_clean.xlsx')

def pre_process_stem_sentence(sentence):
    stemmer = PorterStemmer()
    sentence = re.sub('[^a-zA-Z]', ' ', str(sentence))
    sentence = sentence.lower()
    sentence = sentence.split()
    sentence = [stemmer.stem(word) for word in sentence if
                (word not in set(stopwords.words('english'))) and (len(word) > 2)]
    sentence = ' '.join(sentence)
    return sentence


def prepareEvaluationData(data, vocab2index, mean_words, stemmed=True, targetColumn='TEXT_CONTENT'):

    # Zero-numbering the labels
#     data['DOCUMENT_TYPE'] = classToLabels(data['DOCUMENT_TYPE'])
    if not stemmed:
        data['text'] = None
        for index, row in tqdm(data.iterrows()):
            data.at[index, 'text'] = pre_process_stem_sentence(row[targetColumn])

    data = data[data.text.str.contains(' ', na=False)]
    data['encoded'] = data['text'].apply(lambda x: np.array(encode_sentence(x, vocab2index, mean_words)))
    # data1 = data[data['DOCUMENT_TYPE']]
    return data
    

def evaluate_model(X,y,model,device,l=1):
    dataset = DocumentDataset(X,y)
    train_dl = DataLoader(dataset, batch_size=500, shuffle=True)
    # Metrics
    y_true = []
    y_pred = []
    with torch.no_grad():
        for inputs, classes, i in train_dl:
            inputs = inputs.long().to(device)
            classes = classes.long().to(device)
#             print(inputs.size())
#             print(classes.size())
            if l>0:
                temp = model(inputs, 1)
            else:
                temp = model(inputs)

            _, outputs = torch.max(temp, 1)
            for (pred, actual) in itertools.zip_longest(outputs, classes):
                y_pred.append(pred.item())
                y_true.append(actual.item())

    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))
    return y_true, y_pred

In [17]:
def loadModel(model_path, vector_path):
    logger.info('Initialising models!')
    with open(vector_path) as f:
        data = f.read()
    vector_config = json.loads(data)
    logger.info('vector_config loaded.')

    length_of_sentence = vector_config['mean_words']
    embedding_dim = vector_config['embedding_dim']
    hidden_dim = vector_config['hidden_dim']
    n_classes = vector_config['n_classes']
    vocab2index = vector_config['word_map']

    model = torch.load(model_path, map_location=device)
    # classifier_path = './' + model_path
    # checkpoint = torch.load(classifier_path, map_location=device)
    # model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print('model loaded')

    return model, length_of_sentence, vocab2index


def loadWord2Vec(word2vec_path):
    wv = KeyedVectors.load(word2vec_path, mmap='r')
    word_map = {
        word: idx
        for idx, word in enumerate(wv.key_to_index, start=2)
    }
    word_map[""] = 0
    word_map['UNK'] = 1

    word_vectors = pd.np.insert(
        wv.vectors,
        0,
        pd.np.random.uniform(wv.vectors.min(), wv.vectors.max(), 300),
        axis=0
    )

    word_vectors = pd.np.insert(
        word_vectors,
        0,
        pd.np.zeros(300),
        axis=0
    )
    word_vectors = torch.FloatTensor(word_vectors)

    return word_vectors, word_map

In [18]:
def getModel(model_path, embedding_dim, hidden_dim, n_classes, weights, device=torch.device("cuda:1" if torch.cuda.is_available() else "cpu")):
    model = LstmW2VNet(embedding_dim, hidden_dim, n_classes, weights)
    classifier_path = './' + model_path
    checkpoint = torch.load(classifier_path, map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    print('model loaded')
    return checkpoint

In [19]:
# data = pd.read_excel('misc_df.xlsx')
# data.info()

In [20]:
# data = pd.read_excel('misc_df.xlsx')
# data.info()

In [21]:
data = pd.read_excel('eval_data.xlsx')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14245 entries, 0 to 14244
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SHIPMENT_ID    14245 non-null  object 
 1   DOCUMENT_ID    14245 non-null  object 
 2   DOCUMENT_NAME  0 non-null      float64
 3   DOCUMENT_TYPE  14245 non-null  int64  
 4   CREATED_DATE   14245 non-null  int64  
 5   TEXT_CONTENT   11294 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 667.9+ KB


In [22]:
# data=data.loc[data['DOCUMENT_TYPE']!=0]
# data.info()

In [23]:
# data=data.loc[data['UPDATED_DOC_TYPE']!=0]
# data.info()

In [24]:
# data['DOCUMENT_TYPE'].value_counts()

In [25]:
# data_train = pd.read_excel('stemmed_balanced_relevant_threeKeyword.xlsx')
# data.rename(columns={'words': 'TEXT_CONTENT'}, inplace=True)


In [26]:
data['text']=''
from pathlib import Path
modelPath = Path('classification_folder')

data = data[['SHIPMENT_ID', 'DOCUMENT_ID',
       'DOCUMENT_TYPE', 'CREATED_DATE', 'TEXT_CONTENT', 'text']]

In [27]:
# from pathlib import Path
# modelPath = Path('classification_folder')

# data = data[['SHIPMENT_ID', 'DOCUMENT_ID',
#        'DOCUMENT_TYPE', 'TEXT_CONTENT', 'text','ocr','ocr_stemmed']]

In [28]:
# from pathlib import Path
# modelPath = Path('classification_folder')

# data = data[['SHIPMENT_ID', 'DOCUMENT_ID',
#        'DOCUMENT_TYPE', 'CREATED_DATE', 'TEXT_CONTENT', 'text', 'UPDATED_DOC_TYPE']]

In [29]:
from gensim.models import KeyedVectors

In [30]:
import logging
logging.basicConfig(format='%(asctime)s,%(msecs)d %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s',
                    datefmt='%Y-%m-%d:%H:%M:%S', level=logging.INFO)
logger = logging.getLogger(__name__)

In [31]:
model_path = str(modelPath/'classificationModel_baseDataPlusJan8HashedAndStemmedOversampled_ForSplit-3.pth')
vector_path = str(modelPath/'vector_v1_baseDataPlusJan8HashedAndStemmedOversampled.txt')
# word2vec_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
word2vec_path = 'word2vec_cleaned_augmented_updated_rf.wordvectors'
word2vecSupport_path = 'word2vec_cleaned_augmented_updated_rf.wordvectors.vectors.npy'

# if get_check_point_file_from_s3(model_path, vector_path, word2vec_path, word2vecSupport_path):
# if True:
# weights, vocab2index1 = loadWord2Vec(word2vec_path)

In [32]:
# model_path = str(modelPath/'classificationModel_base+Oct_augmented_word2vec_300_7_stemmed_cv_no_misc_w2v_wo_misc_ForSplit-1.pth')
# vector_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed_cv_no_misc.txt')
# # word2vec_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
# word2vec_path = 'word2vec_augmented_augmented_updated_nlp_done_6e-5_300_win_7_wo_misc.wordvectors'
# word2vecSupport_path = 'word2vec_augmented_augmented_updated_nlp_done_6e-5_300_win_7_wo_misc.wordvectors.vectors.npy'

# # if get_check_point_file_from_s3(model_path, vector_path, word2vec_path, word2vecSupport_path):
# # if True:
# weights, vocab2index1 = loadWord2Vec(word2vec_path)

In [33]:
# model_path = str(modelPath/'classificationModel_base+Oct_augmented_word2vec_300_7_stemmed_cv_ForSplit-4.pth')
# vector_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
# # word2vec_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
# word2vec_path = 'word2vec_augmented_augmented_updated_nlp_done_stemmed_6e-5_300_win_7.wordvectors'
# word2vecSupport_path = 'word2vec_augmented_augmented_updated_nlp_done_stemmed_6e-5_300_win_7.wordvectors.vectors.npy'

# # if get_check_point_file_from_s3(model_path, vector_path, word2vec_path, word2vecSupport_path):
# # if True:
# weights, vocab2index1 = loadWord2Vec(word2vec_path)

In [34]:
# model_path = str(modelPath/'classificationModel_base+Oct_augmented_word2vec_300_5_stemmed_cv_ForSplit-4.pth')
# vector_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_5_stemmed_cv.txt')
# # word2vec_path = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
# word2vec_path = 'word2vec_augmented_augmented_updated_nlp_done_stemmed_6e-5_300_win_5.wordvectors'
# word2vecSupport_path = 'word2vec_augmented_augmented_updated_nlp_done_stemmed_6e-5_300_win_5.wordvectors.vectors.npy'

# # if get_check_point_file_from_s3(model_path, vector_path, word2vec_path, word2vecSupport_path):
# # if True:
# weights, vocab2index1 = loadWord2Vec(word2vec_path)

In [35]:
# weights.shape

In [36]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")


In [37]:
model1, mean_words1, vocab2index1 = loadModel(model_path, vector_path)

2023-02-15:19:02:13,375 INFO     [<ipython-input-17-b423e04bc96e>:2] Initialising models!
2023-02-15:19:02:13,436 INFO     [<ipython-input-17-b423e04bc96e>:6] vector_config loaded.


model loaded


In [38]:
# vector_path1 = str(modelPath/'vector_v1_base+Oct_augmented_word2vec_300_7_stemmed.txt')
# model_path1 = str(modelPath/'classificationModel_base+Oct_augmented_word2vec_300_7_stemmed_cv_ForSplit-4.pth')

# device1 = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")


# model1.to(device1)

# print(model1)

In [39]:
# print(device1)

In [40]:
# print(model1)

In [41]:
# vector_path2 = str(modelPath/'vector_v1_base+Oct_oversampled.txt')
# model_path2 = str(modelPath/'classificationModel_base+Oct_oversampled_ForSplit-1.pth')

# device2 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# model2, vocab2index2, mean_words2 = load_model(model_path2, vector_path2, False)
# model2.to(device2)

# print(model2)


In [42]:
# model2, vocab2index, mean_words = load_model_local('best_checkpoint_10-4_181.pth')
# print(model2)
# text = data

# for index, row in data.iterrows():
#     data.at[index, 'TEXT_CONTENT'] = ' '.join(eval(row['TEXT_CONTENT']))

data.dropna(inplace=True)

2023-02-15:19:02:20,226 INFO     [utils.py:129] Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2023-02-15:19:02:20,227 INFO     [utils.py:141] NumExpr defaulting to 8 threads.


In [43]:
# text = data_train[data_train['SHIPMENT_ID'] == 'KX-Z2J7-1']['text'].any()
# classify_local(model1, vocab2index, mean_words, text)

In [44]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11294 entries, 0 to 14244
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   SHIPMENT_ID    11294 non-null  object
 1   DOCUMENT_ID    11294 non-null  object
 2   DOCUMENT_TYPE  11294 non-null  int64 
 3   CREATED_DATE   11294 non-null  int64 
 4   TEXT_CONTENT   11294 non-null  object
 5   text           11294 non-null  object
dtypes: int64(2), object(4)
memory usage: 617.6+ KB


In [45]:
# def classTextToLabels(y):
#     zero_numbering = {'MISC': 0, 'CI': 1, 'BOL': 2, 'PL': 3, 'AN': 5, 'CD': 6, 'ISF':12}
#     return y.apply(lambda x: zero_numbering[x])

# data['labels'] = classToLabels(classTextToLabels(data['labels']))


In [46]:
data.head()

,SHIPMENT_ID,DOCUMENT_ID,DOCUMENT_TYPE,CREATED_DATE,TEXT_CONTENT,text
0,KX-A0A7-2,ZpuXXmlOoTdrPUHPRdPx,4,1675710890357,"KERRY APEX MARITIME CO . , INC . APEX 877 MAHL...",
1,KX-A0A7-2,nG4042lufSL40Y2Lsjko,2,1675710890488,Shipper / Exporter ( complete name and address...,
6,KX-A0I4-723,Yt8hWdQ4lEHzQiw2lOqy,4,1676411701496,"Logistics Inc. Dyna 18601 S. Susana Rd . , Ran...",
7,KX-A0I4-724,n3ooWWRD0gwAsddVrEFr,4,1676411534362,"Logistics Inc. Dyna 18601 S. Susana Rd . , Ran...",
10,KX-A0I4-725,T4QTy6siuMMuEt3F9MoC,4,1676412423178,"Logistics Inc. Dyna 18601 S. Susana Rd . , Ran...",


In [47]:
data = data[data.TEXT_CONTENT.str.contains(' ', na=False)]

In [48]:
# data.tail()

In [49]:
data1 = prepareEvaluationData(data, vocab2index1, mean_words1, stemmed=False)
# data2 = prepareEvaluationData(data, vocab2index2, mean_words2, False)

# for index, row in data1.iterrows():
#     if row['DOCUMENT_TYPE'] == 8:
#         data1.at[index, 'DOCUMENT_TYPE'] = 7

# for index, row in data2.iterrows():
#     if row['DOCUMENT_TYPE'] == 8:
#         data1.at[index, 'DOCUMENT_TYPE'] = 7

11293it [09:53, 19.03it/s]
/home/ubuntu/anaconda3/envs/pytorch_latest_p36/lib/python3.6/site-packages/ipykernel/__main__.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [50]:
# data1

In [51]:
# data1.head()

In [52]:
# data1

In [53]:
data1['DOCUMENT_TYPE'].value_counts()

1    2994
3    2135
2    1812
4    1791
0    1660
6     459
5     325
8      47
7      33
Name: DOCUMENT_TYPE, dtype: int64

In [54]:
# data1=data1.loc[data1['DOCUMENT_TYPE']!='rtyhjgsdfghjgf']

In [55]:
# data1=data1.loc[data1['DOCUMENT_TYPE']!=30]

In [56]:
# data1=data1.loc[data1['UPDATED_DOC_TYPE']!=30]

In [57]:
data1['DOCUMENT_TYPE']=data1['DOCUMENT_TYPE'].apply(int)

/home/ubuntu/anaconda3/envs/pytorch_latest_p36/lib/python3.6/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':


In [58]:
# data1.loc[data1['DOCUMENT_TYPE']]

In [59]:
data1['labels'] = data1['DOCUMENT_TYPE']
data2 = data1

/home/ubuntu/anaconda3/envs/pytorch_latest_p36/lib/python3.6/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':


In [60]:
# # data1[data1['DOCUMENT_TYPE'] == 7]['REMOTE_LOCATION']
# data1['labels'] = classToLabels(data1['DOCUMENT_TYPE'])
# data2 = data1

In [61]:
# # data1[data1['DOCUMENT_TYPE'] == 7]['REMOTE_LOCATION']
# data1['labels'] = classToLabels(data1['UPDATED_DOC_TYPE'])
# data2 = data1

In [62]:
X = list(data1['encoded'])
y = list(data1['labels'])

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

In [63]:
_, predictions = evaluate_model(X,y,model1,device)

[[1500  104   21    8   11   10    1    5    0]
 [  74 2877    2   33    4    0    4    0    0]
 [  21    3 1780    0    8    0    0    0    0]
 [  21   28    0 2086    0    0    0    0    0]
 [  35    2   15    6 1732    0    0    0    1]
 [   7    0    8    0    0  310    0    0    0]
 [   1    0    3    0    0    0  455    0    0]
 [   0    0    0    0    0    0    0   33    0]
 [   0    0    0    0    0    0    0    0   47]]
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1660
           1       0.95      0.96      0.96      2994
           2       0.97      0.98      0.98      1812
           3       0.98      0.98      0.98      2135
           4       0.99      0.97      0.98      1791
           5       0.97      0.95      0.96       325
           6       0.99      0.99      0.99       459
           7       0.87      1.00      0.93        33
           8       0.98      1.00      0.99        47

    accuracy             

In [65]:
# model_path = str(modelPath/'classificationModel_cleaned_augmented_word2vec_jan_23_wm_and_model_saved_ForSplit-1.pth')
# vector_path = str(modelPath/'vector_v1_cleaned_augmented_word2vec_jan_23_wm_and_model_saved.txt')
# model2, mean_words2, vocab2index2 = loadModel(model_path, vector_path)

In [66]:
# _, predictions = evaluate_model(X,y,model2,device1)

In [67]:
# data

In [68]:
# data['predictions']=np.array(predictions)

In [69]:
# data.loc[data['DOCUMENT_TYPE']!=data['predictions']]

In [70]:
# def evaluate_model(X,y,model,device,l=1):
#     dataset = DocumentDataset(X,y)
#     train_dl = DataLoader(dataset, batch_size=500, shuffle=True)
#     # Metrics
#     y_true = []
#     y_pred = []
#     with torch.no_grad():
#         for inputs, classes, i in train_dl:
#             inputs = inputs.long().to(device)
#             classes = classes.long().to(device)
# #             print(inputs.size())
# #             print(classes.size())
#             if l>0:
#                 temp = model(inputs, 1)
#             else:
#                 temp = model(inputs)

#             _, outputs = torch.max(temp, 1)
#             for (pred, actual) in itertools.zip_longest(outputs, classes):
#                 y_pred.append(pred.item())
#                 y_true.append(actual.item())

#     print(confusion_matrix(y_true, y_pred))
#     print(classification_report(y_true, y_pred))
#     return y_true, y_pred

In [71]:
dataset = DocumentDataset(X,y)

In [72]:
train_dl = DataLoader(dataset, batch_size=500, shuffle=True)

In [73]:
# train_dl

In [74]:
# Metrics
y_true = []
y_pred = []

In [75]:
results_df=pd.DataFrame(columns=['inputs','classes','temp','output'])

In [76]:
with torch.no_grad():
    for inputs, classes, i in train_dl:
        inputs = inputs.long().to(device)
        classes = classes.long().to(device)
#         print(inputs.size())
#         print(classes.size())
        temp = model1(inputs, 1)
        _, outputs = torch.max(temp, 1)
        inputs_array=inputs.cpu().numpy()
        classes_array=classes.cpu().numpy()
        temp_array=temp.cpu().numpy()
        outputs_array=outputs.cpu().numpy()
        temp_df=pd.DataFrame(columns=['inputs','classes','temp','output'])
        temp_df['inputs']=list(inputs_array)
        temp_df['classes']=classes_array
        temp_df['temp']=list(temp_array)
        temp_df['output']=outputs_array
        results_df=pd.concat([results_df,temp_df])
#         print(temp_df)
        
#         print(inputs_array)
#         print(type(classes))
#         print(type(temp))
#         print(type(outputs))

#         break
#         print(outputs.size())
#         print(temp.size())
#         for otpt in outputs:
#             print(otpt)
#             break

In [77]:
# inp=data1['encoded'].iloc[0][0]
# inp

In [78]:
# results_df['inputs'].iloc[2].shape

In [79]:
# transformed_array=[]
# for ipt in 

In [80]:
# inp.shape

In [82]:
results_df.to_excel(r'results_classificationModel_baseDataPlusJan8HashedAugmentedAndStemmed_ForSplit-4.xlsx',index=False)
data1.to_excel(r'evalData_classificationModel_baseDataPlusJan8HashedAugmentedAndStemmed_ForSplit-4.xlsx',index=False)

In [156]:
preds=[]
for encoding in tqdm(data1['encoded']):
    pred='error'
    encoding=encoding[0]
#     print(encoding.shape)
    for i in range(len(results_df)):
        row=results_df.iloc[i]
    #     print(row['inputs']==inp)
        if (np.array_equal(row['inputs'], encoding)):
            pred=row['output']
    preds.append(pred)
#     break

  1%|          | 94/11256 [03:08<6:13:00,  2.01s/it]


KeyboardInterrupt: 

In [154]:
len(preds)

4

In [145]:

for i in range(len(results_df)):
    row=results_df.iloc[0]
#     print(row['inputs']==inp)
    print(np.array_equal(row['inputs'], inp))
    break

False


In [138]:
for 

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [66]:
# results_df.head()

In [60]:
results_df.to_excel(r'results_df_wv_wo_misc.xlsx',index=False)

In [94]:
outputs_array

array([4, 2, 1, 4, 4, 1, 1, 1, 4, 3, 1, 4, 2, 1, 1, 4, 7, 4, 1, 1, 2, 4,
       4, 2, 7, 1, 2, 2, 4, 3, 7, 1, 2, 4, 4, 7, 4, 2, 1, 2, 6, 2, 1, 3,
       6, 2, 8, 1, 3, 1, 2, 4, 2, 2, 3, 1, 3, 7, 4, 4, 3, 2, 1, 5, 4, 4,
       4, 2, 4, 1, 2, 1, 3, 4, 4, 3, 4, 3, 2, 7, 3, 2, 4, 3, 1, 2, 3, 3,
       4, 4, 4, 4, 7, 2, 1, 1, 3, 1, 4, 4, 4, 3, 3, 3, 6, 7, 3, 1, 4, 1,
       3, 3, 1, 1, 4, 4, 1, 4, 4, 2, 4, 4, 2, 1, 1, 4, 4, 1, 4, 4, 7, 2,
       6, 4, 7, 5, 6, 3, 4, 7, 1, 3, 3, 7, 1, 3, 2, 4, 6, 1, 2, 4, 4, 1,
       3, 7, 5, 1, 3, 1, 2, 1, 2, 4, 7, 2, 3, 2, 1, 4, 7, 1, 3, 7, 1, 7,
       1, 2, 3, 1, 2, 1, 5, 1, 2, 4, 1, 1, 4, 1, 1, 3, 2, 1, 3, 4, 6, 2,
       2, 1, 4, 5, 1, 4, 7, 1, 4, 7, 1, 7, 1, 3, 6, 3, 2, 3, 1, 4, 2, 1,
       7, 2, 4, 4, 4, 4, 4, 5, 7, 3, 3, 3, 6, 4, 2, 1, 3, 4, 2, 3, 3, 4,
       4, 1, 2, 4, 4, 1, 4, 2, 1, 2, 7, 1, 4, 3, 6, 4, 4, 4, 3, 3, 4, 3,
       1, 2, 2, 3, 3, 4, 1, 4, 3, 4, 1, 6, 7, 6, 1, 1, 7, 4, 2, 1, 1, 3,
       7, 2, 4, 4, 3, 6, 1, 2, 6, 1, 2, 7, 1, 1, 4,

In [54]:
with torch.no_grad():
    for inputs, classes, i in train_dl:
        inputs = inputs.long().to(device)
        classes = classes.long().to(device)
#             print(inputs.size())
#             print(classes.size())
        if l>0:
            temp = model(inputs, 1)
        else:
            temp = model(inputs)

0

[3, 3, 4, 6, 7]

In [52]:
_, predictions = evaluate_model(X,y,model1,device1)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# X = list(data2['encoded'])
# y = list(data2['DOCUMENT_TYPE'])

# X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

# _, predictions2 = evaluate_model(X,y,model2,device2,-1)

[[   0 1111  813  805 1227  152  220  547   11]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00    4886.0
           1       0.00      0.00      0.00       0.0
           2       0.00      0.00      0.00       0.0
           3       0.00      0.00      0.00       0.0
           4       0.00      0.00      0.00       0.0
           5       0.00      0.00      0.00       0.0
           6       0.00      0.00      0.00       0.0
           7       0.00      0.00      0.00       0.0
           8       0.00      0.00      0.00       0.0

    accuracy             

/home/ubuntu/anaconda3/envs/pytorch_latest_p36/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ubuntu/anaconda3/envs/pytorch_latest_p36/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [53]:
misc_eval_df=pd.DataFrame(columns=['input','label','pred','score'])

In [54]:
misc_eval_df['input']=data1['encoded']
misc_eval_df['label']=data1['labels']
misc_eval_df['pred']=np.array(predictions)

In [57]:
misc_eval_df

,input,label,pred,score
0,"[[43, 182, 54, 186, 731, 11464, 11464, 555, 16...",0,1,NaN
1,"[[2223, 1480, 140, 125, 140, 125, 1056, 6296, ...",0,1,NaN
2,"[[74, 186, 2223, 797, 3367, 964, 14245, 2223, ...",0,1,NaN
3,"[[1420, 4987, 37, 3249, 445, 1755, 45, 113, 33...",0,3,NaN
4,"[[1420, 4987, 37, 3249, 445, 1755, 45, 113, 33...",0,4,NaN
...,...,...,...,...
4913,"[[5373, 132, 120, 454, 534, 537, 573, 11386, 1...",0,3,NaN
4914,"[[11682, 1939, 11180, 4512, 5325, 321, 898, 44...",0,4,NaN
4915,"[[11682, 1939, 11180, 4512, 5325, 321, 898, 44...",0,4,NaN
4916,"[[11682, 1939, 11180, 4512, 5325, 321, 898, 44...",0,1,NaN


In [63]:
x=torch.from_numpy(X[0])

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint8, and bool.

In [60]:
y_hat = model1(x, 1)

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not numpy.ndarray

In [ ]:
logger.info(y_hat)

In [23]:
# data.to_excel('verified_data_testSet.xlsx')
data.labels.value_counts()

AttributeError: 'DataFrame' object has no attribute 'labels'

In [ ]:
# data.info()

In [ ]:
# data = prepareEvaluationData(data)
# # X = list(data['encoded'])
# # y = list(data['DOCUMENT_TYPE'])

# X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)
data = data[data.text.str.contains(' ', na=False)]

# # Separate test and validation.
# # X_test, X_valid, y_test, y_valid = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=True, random_state=42, stratify=y_valid)

preds = []
for index, row in data.iterrows():
    text = row['text']
    pred = classify_local(model1, vocab2index, mean_words, text)
    data.loc[index, 'prediction'] = pred
    preds.append(pred)
    print(index)
# temp = data[data['prediction'] != data['DOCUMENT_TYPE']]
data.to_excel('complete_predictions_march22_misc_added.xlsx')

In [ ]:
data

In [ ]:
data_train.to_excel('predictions_threeKeywordModel_complete.xlsx')

In [ ]:
data = prepareEvaluationData(data)
X = list(data['encoded'])
y = list(data['DOCUMENT_TYPE'])

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

# Separate test and validation.
# X_test, X_valid, y_test, y_valid = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=True, random_state=42, stratify=y_valid)

_, predictions = evaluate_model(X,y,model1)

In [ ]:
# changing 0-numbering to original classes
def labelToClass(y):
    zero_numbering = {0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 6, 6: 12}
    return zero_numbering[y]

preds = []
for i in predictions:
    preds.append(labelToClass(i))


In [ ]:

preds = pd.Series(preds)
data['predictions'] = preds

In [ ]:
data.to_excel('stemmed_balanced_relevant_zeroKeyword.xlsx')

In [ ]:
model3, vocab2index, mean_words = load_model_local('best_checkpoint_10-3.pth')
evaluate_model(X_valid,y_valid,model3)

# print('prediction complete')
# data.to_excel('temp_relevant_balanced_predictions.xlsx')

In [ ]:
model4, vocab2index, mean_words = load_model_local('best_checkpoint_10-4.pth')
# y_pred = preds
# y_true = data['DOCUMENT_TYPE'].to_numpy()


In [ ]:
from sklearn.metrics import confusion_matrix


In [ ]:
model

In [ ]:
torch.save({
                'epoch': i + 1,
                'model_state_dict': model.state_dict(),
            }, 'best_checkpoint_10-4_136.pth')

In [ ]:
## Plot precision-recall curve
for i in range(len(classes)):
    precision, recall, thresholds = metrics.precision_recall_curve(
                 y_test_array[:,i], predicted_prob[:,i])
    ax[1].plot(recall, precision, lw=3, 
               label='{0} (area={1:0.2f})'.format(classes[i], 
                                  metrics.auc(recall, precision))
              )
ax[1].set(xlim=[0.0,1.05], ylim=[0.0,1.05], xlabel='Recall', 
          ylabel="Precision", title="Precision-Recall curve")
ax[1].legend(loc="best")
ax[1].grid(True)
plt.show()
